## Setup runner

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="xr client utilities example")
imd_runner.load(0)
imd = imd_runner.app_server.imd

## Prepare universe
We use this to get atom data more easily for the "info mode" later.

In [2]:
from nanover.mdanalysis import frame_data_to_mdanalysis

universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

## XR client notification
The XR client defines a remote command to send it notifications (displayed over the controller). We define a function to find all of them and call them to notify all clients.

In [3]:
def notify_all(message):
    for command in imd_runner.app_server.commands:
        if "/notify" in command:
            imd_runner.app_server.run_command(command, dict(message=message))

## Modes & interaction triggers
This code snippet defines a basic system of "modes" that can be switched between by the user, providing different sets of behavior in response to interactions starting and stopping.

In [4]:
from nanover.imd.imd_state import ParticleInteraction

class Mode:
    active = None

    @staticmethod
    def add_to_runner(imd_runner):
        def on_interaction_started(*, key: str, interaction: ParticleInteraction):
            Mode.active.active.on_interaction_started(key=key, interaction=interaction)

        def on_interaction_stopped(*, key: str, interaction: ParticleInteraction):
            Mode.active.on_interaction_stopped(key=key, interaction=interaction)

        def on_interaction_updated(*, key: str, interaction: ParticleInteraction):
            Mode.active.on_interaction_updated(key=key, interaction=interaction)

        imd_runner.app_server.imd.interaction_started.add_callback(on_interaction_started)
        imd_runner.app_server.imd.interaction_stopped.add_callback(on_interaction_stopped)
        imd_runner.app_server.imd.interaction_updated.add_callback(on_interaction_updated)
        Mode.enter()

    @classmethod
    def enter(cls):
        Mode.active = cls()
        notify_all(f"ENTERED {cls.__name__}")

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        pass

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        pass

    def on_interaction_updated(self, *, key: str, interaction: ParticleInteraction):
        pass

Mode.add_to_runner(imd_runner)

In [5]:
class NormalMode(Mode):
    pass

imd_runner.app_server.register_command("user/normal mode", NormalMode.enter)

## Info mode
Defining a mode where every user interaction triggers a notification in the XR client with information about the interacted atom from MDAnalysis:

In [6]:
class InfoMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        atom = universe.atoms[interaction.particles[0]]
        notify_all(str(atom))

imd_runner.app_server.register_command("user/info mode", InfoMode.enter)

## Recording
Defining commands that start and stop recording to a file:

In [7]:
from nanover.websocket.record import record_from_runner, BackgroundRecordingContext

RECORDING_PATH = None
RECORDING_COUNT = 0
RECORDER: BackgroundRecordingContext | None = None

def start_recording():
    global RECORDER, RECORDING_COUNT, RECORDING_PATH
    RECORDING_PATH = f"RECORDING-{RECORDING_COUNT}-{imd_runner.simulation.name}.nanover.zip"
    RECORDER = record_from_runner(imd_runner, RECORDING_PATH)
    RECORDING_COUNT += 1
    notify_all(f"STARTED RECORDING to {RECORDING_PATH}")

def stop_recording():
    if RECORDER is not None:
        RECORDER.close()
        notify_all(f"FINISHED RECORDING to {RECORDING_PATH}")

imd_runner.app_server.register_command("user/recording/start", start_recording)
imd_runner.app_server.register_command("user/recording/stop", stop_recording)

In [8]:
import numpy as np
import numpy.typing as npt
from dataclasses import dataclass, field
from concurrent.futures import ThreadPoolExecutor
from nanover.utilities.cli import CancellationToken

@dataclass(kw_only=True, unsafe_hash=True)
class Group:
    particles: list[int] = field(default_factory=list)
    path: list[list[float]] = field(default_factory=list)
    progress = 0


def get_path_length(path: list[npt.NDArray[np.float32]]) -> float:
    length = 0

    prev = path[0]
    for next in path[1:]:
        length += np.linalg.norm(next - prev)
        prev = next

    return length


def get_point_on_path_at_distance(path: list[npt.NDArray[np.float32]], distance: float) -> npt.NDArray[np.float32] | None:
    prev = path[0]
    for next in path[1:]:
        length = np.linalg.norm(next - prev)
        prev = next
        if distance > length:
            distance -= length
            continue
        else:
            u = distance / length
            p = prev + (next - prev) * u
            return p

    return None


class Replayer:
    def __init__(self):
        self._threads = ThreadPoolExecutor(max_workers=1)
        self._cancellation = CancellationToken()
        self._task = None

    def start(self):
        if self._task is not None:
            return

        publisher = imd_runner.app_server.frame_publisher
        imd = imd_runner.app_server.imd
        stream = publisher.subscribe_latest_frames(frame_interval=0, cancellation=self._cancellation)

        def run():
            distance = 0

            for frame in stream:
                distance += .01

                for i, group in enumerate(GROUPS):
                    key = f"interaction.REPLAYER.{i}"
                    #centroid = np.average(frame.particle_positions[group.particles], axis=0)

                    target = get_point_on_path_at_distance(group.path, distance)

                    if target is not None:
                        imd.insert_interaction(key, ParticleInteraction(
                            particles=group.particles,
                            position=list(target),
                            interaction_type="spring",
                            scale=500,
                        ))
                        notify_all(str(target))
                    else:
                        imd.remove_interaction(key)

        self._task = self._threads.submit(run)

    def stop(self):
        self._cancellation.cancel()
        self._threads.shutdown(wait=True)


def clone_omni_omm(omni_omm):
    """
    Create an independent copy of a given NanoVer OpenMMSimulation.
    """
    from nanover.openmm import serializer

    # get the underlying openmm simulation object
    omm = omni_omm.simulation

    # clone it by serializing and then deserializing
    data = serializer.serialize_simulation(omm)
    omm_clone = serializer.deserialize_simulation(data)

    # wrap the cloned openmm simulation in OpenMMSimulation
    omni_omm_clone = OpenMMSimulation.from_simulation(omm_clone)

    # give it different name
    omni_omm_clone.name = omni_omm.name + "'"

    return omni_omm_clone


def set_checkpoint():
    global CHECKPOINT, GROUPS
    CHECKPOINT = clone_omni_omm(imd_runner.simulation)
    GROUPS = []
    notify_all("CHECKPOINT SET")


def reset_to_checkpoint():
    global REPLAYER
    if REPLAYER is not None:
        REPLAYER.stop()
        REPLAYER = None
    imd_runner.simulations[0] = CHECKPOINT
    imd_runner.load(0)


def replay_from_checkpoint():
    global REPLAYER

    if CHECKPOINT is None:
        notify_all("NO CHECKPOINT")
    else:
        reset_to_checkpoint()
        notify_all("REPLAYING FROM CHECKPOINT")
        REPLAYER = Replayer()
        REPLAYER.start()


CHECKPOINT = None
REPLAYER: Replayer | None = None
GROUPS: list[Group] = []


def on_interaction_updated(*, key: str, interaction: ParticleInteraction):
    if CHECKPOINT is not None and REPLAYER is None:
        for group in GROUPS:
            if set(group.particles) == set(interaction.particles):
                positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
                centroid = np.average(positions[interaction.particles], axis=0)
                group.path.append(centroid)
                break
        else:
            GROUPS.append(Group(particles=interaction.particles))

imd_runner.app_server.imd.interaction_updated.add_callback(on_interaction_updated)

# add the command to the server,
# commands with names beginning with "user/" are displayed in the VR client
imd_runner.app_server.register_command("user/checkpoint/set", set_checkpoint)
imd_runner.app_server.register_command("user/checkpoint/reset", reset_to_checkpoint)
imd_runner.app_server.register_command("user/checkpoint/replay", replay_from_checkpoint)